# 02 - Run Experiment

Run the full 4-stage debate pipeline in **mock mode** (no API keys, no cost) and inspect a complete trace for one problem.

To use real APIs instead, set `MODE = 'real'` below and make sure your `.env` has the relevant keys.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath('..'))

from src.utils import load_problems
from src.config import DEFAULT_PROBLEMS_FILE, ensure_dirs
from src.pipeline.debate_runner import build_clients, run_problem

ensure_dirs()
MODE = 'mock'          # 'mock' or 'real'
RUN_ID = 'notebook_run'
LIMIT = 5              # run a small subset for a quick demo

problems = load_problems(DEFAULT_PROBLEMS_FILE)[:LIMIT]
clients = build_clients(MODE, problems)
print('Clients:', {k: v.model_name for k, v in clients.items()})

Clients: {'gpt': 'mock-gpt', 'claude': 'mock-claude', 'gemini': 'mock-gemini', 'grok': 'mock-grok'}


In [2]:
from tqdm.auto import tqdm
results = [run_problem(clients, p, run_id=RUN_ID, save=True) for p in tqdm(problems)]
print(f'Ran {len(results)} problems. Traces saved under outputs/runs/{RUN_ID}/')

  0%|          | 0/5 [00:00<?, ?it/s]

Ran 5 problems. Traces saved under outputs/runs/notebook_run/


## Inspect a full debate trace

Walk through every stage for the first problem.

In [3]:
r = results[0]
print('PROBLEM:', r.problem.question)
print('CORRECT ANSWER:', r.problem.correct_answer)
print()
print('Stage 0.5 - Assigned roles:')
print('  Judge :', r.assigned_roles.judge)
print('  Solvers:', r.assigned_roles.solver_slots)

PROBLEM: In how many ways can you tile a 3x8 rectangle with 2x1 dominoes?
CORRECT ANSWER: 153

Stage 0.5 - Assigned roles:
  Judge : gemini
  Solvers: {'solver_1': 'gpt', 'solver_2': 'claude', 'solver_3': 'grok'}


In [4]:
print('Stage 1 - Independent solutions:')
for s in r.initial_solutions:
    print(f'  {s.solver_id} ({s.model_key}): answer={s.final_answer!r} conf={s.confidence}')

print('\nStage 2 - Peer reviews:')
for rev in r.peer_reviews:
    print(f'  {rev.reviewer_id} -> {rev.target_id}: {rev.overall_assessment}')

print('\nStage 3 - Refined answers:')
for ref in r.refinements:
    print(f'  {ref.solver_id} ({ref.model_key}): refined_answer={ref.refined_answer!r} conf={ref.confidence}')

print('\nStage 4 - Judge decision:')
print('  winner:', r.judge_decision.winner, '| reasoning:', r.judge_decision.reasoning)
print('\nFINAL DEBATE ANSWER:', r.debate_final_answer)

Stage 1 - Independent solutions:
  solver_1 (gpt): answer='153' conf=0.74
  solver_2 (claude): answer='157' conf=0.69
  solver_3 (grok): answer='153' conf=0.88

Stage 2 - Peer reviews:
  solver_1 -> solver_2: promising_but_flawed
  solver_1 -> solver_3: promising_but_flawed
  solver_2 -> solver_1: promising_but_flawed
  solver_2 -> solver_3: solid
  solver_3 -> solver_1: promising_but_flawed
  solver_3 -> solver_2: solid

Stage 3 - Refined answers:
  solver_1 (gpt): refined_answer='153' conf=0.74
  solver_2 (claude): refined_answer='153' conf=0.8
  solver_3 (grok): refined_answer='153' conf=0.8

Stage 4 - Judge decision:
  winner: solver_1 | reasoning: solver_1 presents the most rigorous and well-verified solution.

FINAL DEBATE ANSWER: 153


In [5]:
# The raw cached JSON trace (same content saved to outputs/runs/...).
print(json.dumps(r.model_dump(), indent=2)[:2000], '...')

{
  "run_id": "notebook_run",
  "problem": {
    "id": "math_001",
    "category": "mathematical_reasoning",
    "question": "In how many ways can you tile a 3x8 rectangle with 2x1 dominoes?",
    "correct_answer": "153",
    "accepted_answers": [
      "153"
    ],
    "answer_type": "integer",
    "difficulty": "hard",
    "tolerance": null,
    "source_or_notes": "Classic domino tiling recurrence: f(n)=4f(n-2)-f(n-4)."
  },
  "assigned_roles": {
    "judge": "gemini",
    "solvers": [
      "gpt",
      "claude",
      "grok"
    ],
    "solver_slots": {
      "solver_1": "gpt",
      "solver_2": "claude",
      "solver_3": "grok"
    },
    "assessments": [
      {
        "model_key": "gpt",
        "solver_confidence": 0.64,
        "judge_confidence": 0.68,
        "reasoning": "As gpt, I weighed my strengths for problem math_001. Solver confidence 0.64, judge confidence 0.68."
      },
      {
        "model_key": "claude",
        "solver_confidence": 0.81,
        "judge_conf